# Caso práctico: OCR semántico

**Lección 4 · Clase 5.3** — del píxel al JSON: extraer **datos estructurados** desde un formulario escaneado (con letra manuscrita y opciones marcadas a mano) usando un modelo multimodal.

La diferencia con el OCR clásico:

| | OCR clásico | OCR semántico (LLM) |
|---|---|---|
| Salida | Texto plano, carácter por carácter | El **esquema que tú pidas** (JSON) |
| Manuscrito | Frágil | Lo lee en contexto |
| "¿Qué opción está marcada?" | No sabe qué es una opción | Entiende el formulario como documento |

In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q langchain-openai==1.3.5 langchain-core==1.4.9 python-dotenv==1.2.2 pillow==12.3.0 pydantic==2.13.4
from dotenv import load_dotenv
import os

# Carga OPENAI_API_KEY desde .env si existe (local); en Colab usa Secrets.
load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
except Exception:
    pass

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY presente:", True)
else:
    print("⚠️ Falta OPENAI_API_KEY — usa un archivo .env o los Secrets de Colab.")


## El documento

Un formulario real de inspección vehicular: campos manuscritos, opciones Yes/No marcadas con círculos, y observaciones en letra a mano.

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

form_path = Path("car-inspection-form_1.png")
if not form_path.exists():
    import urllib.request
    # En Colab el archivo local no existe: se baja desde el repo (público)
    req = urllib.request.Request(
        "https://raw.githubusercontent.com/josepenam/clases-diplomado-gen-ia/main/"
        "class_5_3_imagenes/leccion4_ocr_semantico/car-inspection-form_1.png",
        headers={"User-Agent": "Mozilla/5.0 (clase-diplomado-gen-ia)"},
    )
    form_path.write_bytes(urllib.request.urlopen(req).read())

display(Image.open(form_path).reduce(2))

## Esquema + salida estructurada

Definimos con **Pydantic** la forma exacta que queremos de vuelta y usamos `with_structured_output()`: el modelo queda **obligado** a responder en ese esquema (por debajo usa los *structured outputs* de OpenAI).

Detalle que importa: los structured outputs exigen **esquemas cerrados** — un `Dict[str, str]` abierto es rechazado por la API. Por eso los campos del formulario se modelan como **lista de pares** `{name, value}`, que es la forma estándar de representar key-values de largo desconocido. El prompt es un template donde la imagen en base64 entra como variable.

In [ ]:
import base64

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


class FormField(BaseModel):
    name: str = Field(description="the field's label as written on the form")
    value: str = Field(description="the field's value as filled in on the form")


class FormAnswer(BaseModel):
    form_fields: list[FormField] = Field(
        description="every piece of information available on the form, as field/value pairs"
    )


def encode_image(image_path: Path) -> str:
    return base64.b64encode(image_path.read_bytes()).decode("utf-8")


def create_form_extractor(model_name: str):
    model = ChatOpenAI(model=model_name)
    prompt = ChatPromptTemplate(
        [
            (
                "system",
                "You are a car inspector expert. Your job is to analyze "
                "the user's form and extract all the relevant information.",
            ),
            (
                "human",
                [
                    {"type": "text", "text": "this is the inspection form you need to analyze"},
                    {
                        "type": "image",
                        "source_type": "base64",
                        "data": "{image_data}",
                        "mime_type": "image/png",
                    },
                ],
            ),
        ]
    )
    return prompt | model.with_structured_output(FormAnswer)

In [ ]:
extractor = create_form_extractor("gpt-5-mini")
resultado = extractor.invoke({"image_data": encode_image(form_path)})

{campo.name: campo.value for campo in resultado.form_fields}

## Discusión

- Compara el JSON con el formulario: ¿leyó bien el manuscrito? ¿acertó en **todas** las opciones marcadas con círculo? Los checkboxes ambiguos son el punto débil típico.
- El esquema manda: si mañana necesitas solo `{make, tag_no, problems}`, cambias el modelo Pydantic y nada más.
- Esto escala a facturas, contratos, licitaciones, fichas médicas… El siguiente nivel es el **parsing agéntico de documentos** (LlamaParse, Azure Document Intelligence): pipelines que deciden por página cómo extraer, verifican su propia salida y reintentan lo dudoso.